In [ ]:
from pathlib import Path                                        
import re                                                    

ROOT = Path.cwd()                                               
while not (ROOT / "knowledge_base").exists() and ROOT != ROOT.parent:  
    ROOT = ROOT.parent
KB_DIR = ROOT / "knowledge_base"                                

SECTION = re.compile(r"^\d+\.\s+\S")                            

def split_file(path):                                           
    lines = path.read_text(encoding="utf-8").splitlines()        
    doc_title = lines[1].strip()                                
    chunks = []                                                  
    title = "Overview"                                           
    body = []                                                  
    for line in lines[2:]:                                    
        if line.strip() == doc_title:                            
            continue                                           
        if SECTION.match(line):                                  
            text = "\n".join(body).strip()                      
            if text:                                            
                chunks.append({"source": path.name, "section": title, "text": doc_title + " - " + title + "\n" + text})  
            title = line.strip()                                
            body = []                                           
        else:                                                 
            body.append(line)                                   
    text = "\n".join(body).strip()                               
    if text:                                                     
        chunks.append({"source": path.name, "section": title, "text": doc_title + " - " + title + "\n" + text})  
    return chunks                                               

chunks = split_file(KB_DIR / "check_in_check_out.txt")          
print(len(chunks), "chunks")                                     
for c in chunks:                                                 
    print("----", c["section"])                                 
    print(c["text"])                                             

5 chunks
---- 1. Check-in
CHECK-IN AND CHECK-OUT POLICY - 1. Check-in
Standard check-in time: 3:00 PM. Guests must present valid identification at check-in (see Identification Policy) and the incidentals deposit is taken.
---- 2. Early Check-in
CHECK-IN AND CHECK-OUT POLICY - 2. Early Check-in
From 12:00 PM early check-in is free, subject to room availability. Before 12:00 PM it costs 50% of the nightly rate. To be sure of a room on arrival before 6:00 AM, the previous night must be booked.
---- 3. Check-out
CHECK-IN AND CHECK-OUT POLICY - 3. Check-out
Standard check-out time: 12:00 PM.
---- 4. Late Check-out
CHECK-IN AND CHECK-OUT POLICY - 4. Late Check-out
Late check-out is subject to availability: until 4:00 PM it costs 1,500 EGP; until 6:00 PM, 50% of the nightly rate; after 6:00 PM a full night is charged. Silver members may stay until 1:00 PM, Gold until 2:00 PM and Platinum until 4:00 PM free of charge, subject to availability.
---- 5. Reception
CHECK-IN AND CHECK-OUT POLICY - 5

In [2]:
import sys
import json
import numpy as np
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "knowledge_base").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from my_rag.chunking import load_model
from my_rag.config import CHUNKS_FILE, VECTORS_FILE

chunks = []
with open(CHUNKS_FILE, encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))
vectors = np.load(VECTORS_FILE).astype(np.float32)
model = load_model()

question = "ممكن أدخل الأوضة بدري؟"
q = model.encode([question], normalize_embeddings=True)[0]
scores = vectors @ q
for i in np.argsort(-scores)[:5]:
    print(round(float(scores[i]), 2), chunks[i]["chunk_id"], "|", chunks[i]["text"][:80].replace("\n", " "))

c:\projects\HotelAI-Intelligent-Agent_Final_Data\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1028 00:10:05.284000 21452 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 48746.33it/s]


0.69 FAQ2561 | Q: ينفع أدخل بدري قبل الميعاد؟ A: الدخول الرسمي الساعة 3 العصر. من 12 الضهر تقدر
0.68 FAQ2562 | Q: هوصل الصبح، أستنى لحد 3 ولا أدخل؟ A: الدخول الرسمي الساعة 3 العصر. من 12 الضه
0.67 FAQ2563 | Q: ممكن early check-in؟ A: الدخول الرسمي الساعة 3 العصر. من 12 الضهر تقدر تدخل ب
0.61 FAQ2556 | Q: أقدر أدخل الأوضة الساعة كام؟ A: الدخول من 3 العصر.
0.6 FAQ3810 | Q: دخلت الأوضة لقيتها مش نضيفة خالص ومش هقدر أقعد فيها A: كلّم الاستقبال فوراً ل


In [3]:
print(len(chunks), vectors.shape)

sources = set()
for c in chunks:
    if c["kind"] == "kb":
        sources.add(c["source"])
print(len(sources), "files have chunks")

empty = 0
for c in chunks:
    if not c["text"].strip():
        empty += 1
print(empty, "empty chunks")

2848 (2848, 1024)
44 files have chunks
0 empty chunks


In [4]:
import csv
from my_rag.config import FAQ_FILE

tests = []
with open(FAQ_FILE, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        if row["split"] == "test" and row["requires_tool"] == "false" and row["source_doc"] != "out_of_scope_policy":
            tests.append(row)
print(len(tests), "test questions")

questions = []
for row in tests:
    questions.append(row["question"])
q_vecs = model.encode(questions, normalize_embeddings=True, batch_size=16).astype(np.float32)
all_scores = q_vecs @ vectors.T

hits = {}
totals = {}
for i, row in enumerate(tests):
    top5 = np.argsort(-all_scores[i])[:5]
    found = False
    for j in top5:
        if chunks[j]["source"] == row["source_doc"]:
            found = True
    lang = row["language"]
    totals[lang] = totals.get(lang, 0) + 1
    if found:
        hits[lang] = hits.get(lang, 0) + 1

for lang in totals:
    print(lang, round(100 * hits.get(lang, 0) / totals[lang]), "%", f"({hits.get(lang, 0)}/{totals[lang]})")
print("ALL", round(100 * sum(hits.values()) / sum(totals.values())), "%")

565 test questions
English 100 % (111/111)
Russian 96 % (110/115)
French 99 % (180/182)
Arabic 95 % (113/119)
Arabic-Egyptian 87 % (33/38)
ALL 97 %


In [5]:
for i, row in enumerate(tests):
    if row["language"] != "Arabic-Egyptian":
        continue
    top5 = np.argsort(-all_scores[i])[:5]
    got = []
    for j in top5:
        got.append(chunks[j]["source"])
    if row["source_doc"] not in got:
        print(row["question"], "| expected:", row["source_doc"], "| got:", got[:3])

عايز late check-out، بكام؟ | expected: pricing_and_rates.txt | got: ['check_in_check_out.txt', 'check_in_check_out.txt', 'check_in_check_out.txt']
بتراجعوا الحجز إزاي قبل الإلغاء؟ | expected: cancellation_process.txt | got: ['no_show_policy.txt', 'booking_modification.txt', 'cancellation_policy.txt']
النزيل اللي جنبي بيدخن في أوضته والريحة داخلة عندي | expected: smoking_policy.txt | got: ['booking_modification.txt', 'booking_modification.txt', 'room_types.txt']
عندكم إيه في منيو الروم سيرفس؟ | expected: in_room_dining_menu.txt | got: ['restaurant.txt', 'restaurant.txt', 'room_service.txt']
الفندق فيه كام أوضة؟ | expected: hotel_information.txt | got: ['booking_policy.txt', 'check_in_check_out.txt', 'safety_and_health.txt']
